In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import os, cv2, random, numpy as np


In [12]:
def corrupt_image(img):
    def add_noise(i):
        return np.clip(i + np.random.normal(0,20,i.shape),0,255).astype(np.uint8)
    def blur(i):
        return cv2.GaussianBlur(i,(5,5),0)
    def lowres(i):
        h,w=i.shape[:2]
        return cv2.resize(cv2.resize(i,(w//3,h//3)),(w,h))
    return random.choice([add_noise,blur,lowres])(img)


In [13]:
class RAImageDataset(Dataset):
    def __init__(self, root_dir, img_size=224):
        self.paths=[os.path.join(root_dir,f) for f in os.listdir(root_dir)]

        self.transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((img_size,img_size))
        ])

    def __len__(self): return len(self.paths)

    def __getitem__(self,idx):
        img=cv2.imread(self.paths[idx])
        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

        corrupted=corrupt_image(img.copy())

        return self.transform(corrupted), self.transform(img)


In [14]:
DATA_PATH = r"D:\Image Recognition\data"

dataset = RAImageDataset(DATA_PATH)
loader = DataLoader(dataset,batch_size=8,shuffle=True)
# print("Dataset:",len(dataset))


In [15]:
class Block(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(a,b,3,1,1),
            nn.BatchNorm2d(b),
            nn.ReLU())
    def forward(self,x): return self.net(x)

class ProcessingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            Block(3,64),Block(64,64),
            nn.MaxPool2d(2),
            Block(64,128),Block(128,128),
            nn.Upsample(scale_factor=2),
            Block(128,64),
            nn.Conv2d(64,3,3,1,1),
            nn.Sigmoid())
    def forward(self,x): return self.net(x)


In [16]:
device="cuda" if torch.cuda.is_available() else "cpu"
print("Using:",device)


Using: cuda


In [17]:
model=ProcessingNet().to(device)
model.load_state_dict(torch.load("baseline_model.pth", map_location=device))
print("Baseline weights loaded")


Baseline weights loaded


In [18]:
teacher=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
teacher=teacher.to(device)
teacher.eval()

normalize = transforms.Normalize(
    mean=[0.485,0.456,0.406],
    std=[0.229,0.224,0.225]
)

print("ResNet50  ready")


ResNet50  ready


In [19]:
image_loss_fn=nn.MSELoss()
class_loss_fn=nn.CrossEntropyLoss()

optimizer=optim.Adam(model.parameters(),lr=1e-4)


In [20]:
lambda_values = [0, 0.01, 0.1, 0.5, 1]

for lambda_recog in lambda_values:

    print(f"\n========== Training ResNet50 with λ = {lambda_recog} ==========\n")

    # Reset model each time
    model = ProcessingNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    #  ResNet50 classifier
    import torchvision.models as models
    teacher = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    teacher = teacher.to(device)
    teacher.eval()

    for epoch in range(5):

        total_loss = 0
        pbar = tqdm(loader)

        for corrupted, clean in pbar:

            corrupted = corrupted.to(device)
            clean = clean.to(device)

            # ----- Forward -----
            restored = model(corrupted)

            # ----- Image Loss -----
            img_loss = image_loss_fn(restored, clean)

            # ----- Recognition Loss -----
            with torch.no_grad():
                teacher_pred = teacher(normalize(clean))
                labels = teacher_pred.argmax(dim=1)

            pred = teacher(normalize(restored))
            recog_loss = class_loss_fn(pred, labels)

            # Combined Loss
            loss = img_loss + lambda_recog * recog_loss

            # ----- Backprop -----
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            pbar.set_postfix(
                total=loss.item(),
                img=img_loss.item(),
                cls=recog_loss.item(),
                lam=lambda_recog
            )

        print(f"Epoch {epoch+1} Loss: {total_loss/len(loader):.4f}")

    
    torch.save(model.state_dict(), f"RA_resnet50_lambda_{lambda_recog}.pth")

    print(f"Saved ResNet50 model for λ = {lambda_recog}")


========== Training ResNet50 with λ = 0 ==========



100%|██████████| 375/375 [03:33<00:00,  1.76it/s, cls=1.61, img=0.00354, lam=0, total=0.00354] 


Epoch 1 Loss: 0.0068


100%|██████████| 375/375 [03:31<00:00,  1.77it/s, cls=1.27, img=0.00291, lam=0, total=0.00291] 


Epoch 2 Loss: 0.0049


...

Epoch 3 Loss: 1.8141


100%|██████████| 375/375 [03:22<00:00,  1.85it/s, cls=2.12, img=0.0166, lam=1, total=2.13]  


Epoch 4 Loss: 1.7615


100%|██████████| 375/375 [03:22<00:00,  1.85it/s, cls=1.92, img=0.0131, lam=1, total=1.94]  

Epoch 5 Loss: 1.7467
Saved ResNet50 model for λ = 1


In [21]:
# torch.save(model.state_dict(),"RA_model_resnet50.pth")
# print("RA ResNet50 model saved")
